# Time Series Forecasting for Energy Analytics - NSP Dataset

## Objective
Build Prophet and LSTM forecasting models for all 5 regions using engineered features

## Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from prophet import Prophet
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"

# Helper functions
model_results = {'Model': [], 'Region': [], 'MAE': [], 'RMSE': [], 'MAPE': []}

def calculate_metrics(y_true, y_pred, model_name, region_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

    model_results['Model'].append(model_name)
    model_results['Region'].append(region_name)
    model_results['MAE'].append(mae)
    model_results['RMSE'].append(rmse)
    model_results['MAPE'].append(mape)

    print(f"{model_name} - {region_name}: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%")
    return mae, rmse, mape

def save_and_show_plot(fig, filename):
    output_path = OUTPUT_DIR / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()

/home/bhavik/Dropbox/edu/smu/winter/data_mining/a6_anomaly_detection/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Engineered Features Dataset

In [ ]:
# Load engineered features with rolling statistics
df = pd.read_csv(OUTPUT_DIR / 'engineered_features.csv', parse_dates=['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# Verify key features exist
required_features = ['consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                     'rolling_mean_168h', 'rolling_std_24h', 'hdd']

print("\nRequired features present:")
for feat in required_features:
    exists = feat in df.columns
    print(f"  {feat}: {'✓' if exists else '✗ MISSING'}")

# Handle region column (check if one-hot encoded or categorical)
if 'region' not in df.columns:
    region_cols = [col for col in df.columns if col.startswith('region_')]
    if region_cols:
        df['region'] = df[region_cols].idxmax(axis=1).str.replace('region_', '')
        print("\nReconstructed 'region' column from one-hot encoding")

print(f"\nRegions: {sorted(df['region'].unique())}")
print("\nSample data:")
df[['timestamp', 'region', 'consumption_kwh', 'consumption_kwh_lag_24h',
    'rolling_mean_168h', 'rolling_std_24h']].head()

Dataset shape: (438240, 30)
Regions: <StringArray>
['Annapolis Valley', 'Cape Breton', 'Halifax', 'Pictou County', 'South Shore']
Length: 5, dtype: str
Date range: 2015-01-02 00:00:00 to 2024-12-31 23:00:00
Columns: ['timestamp', 'region', 'hour', 'day_of_week', 'month', 'year', 'week', 'is_weekend', 'season', 'is_holiday', 'temperature_c', 'feels_like_c', 'humidity_pct', 'wind_speed_kmh', 'precipitation_mm', 'cloud_cover_pct', 'renewable_pct', 'consumption_kwh', 'price_per_kwh', 'grid_load_pct', 'co2_kg', 'customer_type', 'demand_response', 'power_outage', 'anomaly_flag', 'anomaly_type', 'peak_demand_flag', 'consumption_kwh_lag_1h', 'consumption_kwh_lag_24h', 'hdd']


,timestamp,region,hour,day_of_week,month,year,week,is_weekend,season,is_holiday,...,co2_kg,customer_type,demand_response,power_outage,anomaly_flag,anomaly_type,peak_demand_flag,consumption_kwh_lag_1h,consumption_kwh_lag_24h,hdd
0,2015-01-02 00:00:00,Annapolis Valley,0,4,1,2015,1,0,Winter,0,...,65.26,Mixed,0,0,0,Normal,0,125.58,125.53,26.8
1,2015-01-02 01:00:00,Annapolis Valley,1,4,1,2015,1,0,Winter,0,...,60.80,Mixed,0,0,0,Normal,0,114.19,94.93,9.5
2,2015-01-02 02:00:00,Annapolis Valley,2,4,1,2015,1,0,Winter,0,...,59.39,Mixed,0,0,0,Normal,0,113.90,103.48,15.8
3,2015-01-02 03:00:00,Annapolis Valley,3,4,1,2015,1,0,Winter,0,...,74.88,Mixed,0,0,0,Normal,0,96.57,82.12,19.8
4,2015-01-02 04:00:00,Annapolis Valley,4,4,1,2015,1,0,Winter,0,...,81.97,Mixed,0,0,0,Normal,0,112.05,81.56,24.2


## Prophet Forecasting - All Regions with Regressors

Features used as regressors:
- temperature_c, humidity_pct
- consumption_kwh_lag_24h
- hdd (Heating Degree Days)
- is_holiday

We'll forecast each region separately with 80/20 train/test split.

In [ ]:
def forecast_region_prophet(region_name, df_full, max_years=None):
    """Run Prophet forecast for one region"""
    print(f"{'='*70}")
    print(f"Prophet Forecasting: {region_name}")
    print(f"{'='*70}")
    # Filter region
    region_df = df_full[df_full['region'] == region_name].copy()
    region_df = region_df.sort_values('timestamp').reset_index(drop=True)

    # Optionally limit to recent years to keep training time reasonable
    if max_years is not None and len(region_df) > 0:
        hours = int(24 * 365 * max_years)
        if len(region_df) > hours:
            print(f"Truncating to last {max_years} years ({hours} rows) for speed.")
            region_df = region_df.tail(hours).reset_index(drop=True)

    prophet_df = region_df[['timestamp', 'consumption_kwh', 
                            'temperature_c', 'humidity_pct', 
                            'hdd', 'consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                            'rolling_mean_168h', 'rolling_std_24h',
                            'is_holiday']].copy()
    prophet_df.columns = ['ds', 'y', 'temperature', 'humidity', 'hdd',
                         'lag_1h', 'lag_24h', 'rolling_mean_168h', 'rolling_std_24h',
                         'is_holiday']

    # Train/test split (80/20)
    split_idx = int(len(prophet_df) * 0.8)
    train_df = prophet_df[:split_idx].copy()
    test_df = prophet_df[split_idx:].copy()

    print(f"Train: {len(train_df)} rows ({train_df['ds'].min()} to {train_df['ds'].max()})")
    print(f"Test:  {len(test_df)} rows ({test_df['ds'].min()} to {test_df['ds'].max()})")

    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=True, seasonality_mode='multiplicative')
    model.add_regressor('temperature')
    model.add_regressor('humidity')
    model.add_regressor('hdd')
    model.add_regressor('lag_1h')
    model.add_regressor('lag_24h')
    model.add_regressor('rolling_mean_168h')
    model.add_regressor('rolling_std_24h')
    model.add_regressor('is_holiday')

    print('Training Prophet model...')
    model.fit(train_df)

    forecast = model.predict(test_df)

    calculate_metrics(test_df['y'].values, forecast['yhat'].values, 'Prophet', region_name)

    export_df = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    export_df['model'] = 'Prophet'
    export_df['region'] = region_name

    return export_df, model, forecast, test_df

In [ ]:
# Forecast all regions
regions = sorted(df['region'].unique())
print(f"Forecasting {len(regions)} regions: {regions}")

prophet_forecasts = []

for region in regions:
    # limit to last 3 years to keep runtime reasonable (adjustable)
    try:
        forecast_df, model, forecast, test_df = forecast_region_prophet(region, df, max_years=3)
        prophet_forecasts.append(forecast_df)
    except Exception as e:
        print(f"Error running Prophet for {region}: {e}")

# Combine
if prophet_forecasts:
    prophet_combined = pd.concat(prophet_forecasts, ignore_index=True)
    print(f"✓ Prophet forecasting complete: {len(prophet_combined)} predictions")
else:
    print('No Prophet forecasts generated.')